## Architecture experiments overview

All architecture-level changes tried in this project, collected from each model

- Faster R-CNN v1 -> v2 (backbone recipe, BN in FPN / RPN, 4-conv + 1-FC RoI head), with and without scale augmentation
- RetinaNet anchors from the box-size analysis (scale 0.5x, extra aspect ratios), focal-loss and IoU-threshold settings
- RT-DETR: number of queries, matching costs, denoising queries, backbone learning rate, input resolution
- FCOS: centre-sampling radius and CIoU box loss (the three original configurations)

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
ROOT = Path('.').resolve().parent


def compare_folds(table, column, reference):
    pivot = table.pivot_table(index='fold', columns='config', values=column)
    rows = []
    for config in pivot.columns:
        difference = pivot[config] - pivot[reference]
        rows.append(dict(config=config, mean=pivot[config].mean(), std=pivot[config].std(),
                         vs_reference=difference.mean(), folds_better=int((difference > 0).sum())))
    return pd.DataFrame(rows).set_index('config')

## Faster R-CNN v1 -> v2

In [2]:
faster_rows = []
v1_cv = pd.read_csv(ROOT / 'faster_rcnn/final_evaluation/faster_rcnn_folds.csv').query(
    "experiment == 'lr_scheduler' and evaluation_split == 'cv_out_of_fold'").sort_values('fold')
for fold, cv_map in zip(v1_cv.fold, v1_cv['map']):
    faster_rows.append(dict(config='v1 lr_scheduler', fold=int(fold), cv_mAP=cv_map))
v1_official = pd.read_csv(ROOT / 'faster_rcnn/official_val_folds.csv').query("run == 'lr_scheduler'")
for row, official in zip(faster_rows, v1_official.sort_values('fold').mAP):
    row['official_mAP'] = official

In [3]:
for config, csv_name in [('v2 (no scale aug)', 'v2_lr_scheduler_5fold.csv'), ('v2 + scale aug', 'v2_lr_scheduler_scale_aug_5fold.csv')]:
    path = ROOT / 'faster_rcnn/faster_rcnn_v2' / csv_name
    if path.exists():
        for row in pd.read_csv(path).itertuples():
            faster_rows.append(dict(config=config, fold=row.fold, cv_mAP=row.cv_mAP, official_mAP=row.official_mAP))
faster = pd.DataFrame(faster_rows)
display(faster.pivot_table(index='fold', columns='config', values=['cv_mAP', 'official_mAP']).round(4))

cv_mAP                                     official_mAP                                 
config v1 lr_scheduler v2 (no scale aug) v2 + scale aug v1 lr_scheduler v2 (no scale aug) v2 + scale aug
fold                                                                                                    
0               0.2377            0.2672         0.2812          0.1960            0.2123         0.2187
1               0.2208            0.2465         0.2649          0.1858            0.1966         0.2222
2               0.2380            0.2394         0.2545          0.1962            0.2137         0.2219
3               0.2015            0.2354         0.2377          0.1828            0.2135         0.2155
4               0.1571            0.1676         0.1801          0.1935            0.2183         0.2312

In [4]:
print('held-out (CV) mAP')
display(compare_folds(faster, 'cv_mAP', 'v1 lr_scheduler').round(4))
print('official validation mAP')
display(compare_folds(faster, 'official_mAP', 'v1 lr_scheduler').round(4))

held-out (CV) mAP


,mean,std,vs_reference,folds_better
config,,,,
v1 lr_scheduler,0.2110,0.0337,0.0000,0
v2 (no scale aug),0.2312,0.0376,0.0202,5
v2 + scale aug,0.2437,0.0389,0.0326,5


official validation mAP


,mean,std,vs_reference,folds_better
config,,,,
v1 lr_scheduler,0.1909,0.0062,0.000,0
v2 (no scale aug),0.2109,0.0083,0.020,5
v2 + scale aug,0.2219,0.0059,0.031,5


## RetinaNet: anchors and loss settings

In [5]:
retina = pd.concat([pd.read_csv(path) for path in sorted((ROOT / 'retinanet').glob('five_fold_results*.csv'))])
print('held-out (CV) mAP')
display(compare_folds(retina, 'mAP', 'baseline').round(4))
print('official validation mAP')
display(compare_folds(retina, 'official_mAP', 'baseline').round(4))

held-out (CV) mAP


,mean,std,vs_reference,folds_better
config,,,,
anchors_x0.5,0.1772,0.0269,-0.0015,2
anchors_x0.5_ratios5,0.1797,0.0284,0.0010,2
baseline,0.1787,0.0285,0.0000,0
focal_a0.5_g1.5,0.1789,0.0315,0.0002,1
iou_0.4_0.3,0.1475,0.0253,-0.0312,0


official validation mAP


,mean,std,vs_reference,folds_better
config,,,,
anchors_x0.5,0.1593,0.0041,0.0110,5
anchors_x0.5_ratios5,0.1557,0.0026,0.0074,4
baseline,0.1483,0.0048,0.0000,0
focal_a0.5_g1.5,0.1523,0.0055,0.0040,3
iou_0.4_0.3,0.1238,0.0054,-0.0245,0


In [6]:
display(pd.read_csv(ROOT / 'retinanet/anchor_analysis/anchor_coverage.csv').round(3))

,config,objects,n,covered_iou50,mean_best_iou,mean_pos_anchors,zero_pos_share
0,"default (baseline, tuned_focal_loss)",all,222879,0.744,0.622,14.411,0.256
1,"default (baseline, tuned_focal_loss)",small,113572,0.498,0.483,3.330,0.502
2,"default (baseline, tuned_focal_loss)",medium,101437,0.999,0.766,25.638,0.001
3,"default (baseline, tuned_focal_loss)",large,7870,0.997,0.774,29.606,0.003
4,tuned_anchors (0.5x),all,222879,0.957,0.662,6.456,0.043
5,tuned_anchors (0.5x),small,113572,0.921,0.652,6.019,0.079
6,tuned_anchors (0.5x),medium,101437,0.995,0.671,6.870,0.005
7,tuned_anchors (0.5x),large,7870,0.992,0.689,7.427,0.008
8,anchor_intermediate_075 (0.75x),all,222879,0.891,0.676,11.375,0.109
9,anchor_intermediate_075 (0.75x),small,113572,0.786,0.613,6.673,0.214


## RT-DETR: queries, matching, denoising, learning rate, resolution

In [7]:
rtdetr = pd.read_csv(ROOT / 'rt_detr/five_fold_results.csv')
rtdetr = rtdetr.groupby('config').filter(lambda group: group.fold.nunique() == 5)
print('held-out (CV) mAP')
display(compare_folds(rtdetr, 'mAP', 'recipe_warmup_ema').round(4))
print('official validation mAP')
display(compare_folds(rtdetr, 'official_mAP', 'recipe_warmup_ema').round(4))

held-out (CV) mAP


,mean,std,vs_reference,folds_better
config,,,,
lr_backbone_2e-5,0.1724,0.0363,0.0032,4
matching_bbox7_giou3,0.1578,0.0323,-0.0113,0
no_denoising,0.2258,0.0325,0.0567,5
no_denoising_640x1152,0.2602,0.0302,0.0910,5
queries_200,0.1663,0.0342,-0.0029,1
queries_450,0.1697,0.0301,0.0006,3
recipe_warmup_ema,0.1691,0.0363,0.0000,0


official validation mAP


,mean,std,vs_reference,folds_better
config,,,,
lr_backbone_2e-5,0.1561,0.0074,0.0076,4
matching_bbox7_giou3,0.1388,0.0074,-0.0097,0
no_denoising,0.2147,0.0118,0.0661,5
no_denoising_640x1152,0.2360,0.0124,0.0874,5
queries_200,0.1541,0.0095,0.0056,3
queries_450,0.1532,0.0088,0.0046,5
recipe_warmup_ema,0.1485,0.0077,0.0000,0


In [8]:
if 'no_denoising_640x1152' in set(rtdetr.config):
    resolution = rtdetr[rtdetr.config.isin(['no_denoising', 'no_denoising_640x1152'])]
    print('resolution step, relative to no_denoising')
    display(compare_folds(resolution, 'mAP', 'no_denoising').round(4))

resolution step, relative to no_denoising


,mean,std,vs_reference,folds_better
config,,,,
no_denoising,0.2258,0.0325,0.0000,0
no_denoising_640x1152,0.2602,0.0302,0.0344,5


## FCOS: centre-sampling radius and box loss

In [9]:
fcos_folds = pd.read_csv(ROOT / 'fcos/fold_results.csv').rename(columns={'run': 'config'})
display(fcos_folds.groupby('config')[['box_loss', 'center_radius']].first())
print('held-out (CV) mAP')
display(compare_folds(fcos_folds, 'mAP', 'F0_baseline').round(4))
print('AP of small objects (held-out)')
display(compare_folds(fcos_folds, 'AP_small', 'F0_baseline').round(4))

,box_loss,center_radius
config,,
F0_baseline,giou,1.5
F1_ciou,ciou,1.5
F2_radius_1_0,giou,1.0


held-out (CV) mAP


,mean,std,vs_reference,folds_better
config,,,,
F0_baseline,0.1969,0.0318,0.0000,0
F1_ciou,0.1969,0.0314,0.0001,2
F2_radius_1_0,0.1960,0.0303,-0.0008,3


AP of small objects (held-out)


,mean,std,vs_reference,folds_better
config,,,,
F0_baseline,0.0774,0.0340,0.0000,0
F1_ciou,0.0759,0.0340,-0.0016,2
F2_radius_1_0,0.0754,0.0307,-0.0020,2


## All 5-fold configurations in one table

In [10]:
overview = pd.concat([
    faster.groupby('config')[['cv_mAP', 'official_mAP']].mean().rename(index=lambda name: f'Faster R-CNN {name}'),
    retina.groupby('config')[['mAP', 'official_mAP']].mean().rename(columns={'mAP': 'cv_mAP'}).rename(index=lambda name: f'RetinaNet {name}'),
    rtdetr.groupby('config')[['mAP', 'official_mAP']].mean().rename(columns={'mAP': 'cv_mAP'}).rename(index=lambda name: f'RT-DETR {name}'),
    fcos_folds.groupby('config')[['mAP']].mean().rename(columns={'mAP': 'cv_mAP'}).rename(index=lambda name: f'FCOS {name}'),
])
display(overview.sort_values('cv_mAP', ascending=False).round(4))

,cv_mAP,official_mAP
config,,
RT-DETR no_denoising_640x1152,0.2602,0.2360
Faster R-CNN v2 + scale aug,0.2437,0.2219
Faster R-CNN v2 (no scale aug),0.2312,0.2109
RT-DETR no_denoising,0.2258,0.2147
Faster R-CNN v1 lr_scheduler,0.2110,0.1909
FCOS F1_ciou,0.1969,NaN
FCOS F0_baseline,0.1969,NaN
FCOS F2_radius_1_0,0.1960,NaN
RetinaNet anchors_x0.5_ratios5,0.1797,0.1557
